In [7]:
# ============================================================
# Cell 1: 导入必要的第三方库
# ============================================================
import numpy as np      # NumPy: 用于矩阵运算、线性代数求解
import json             # json: 用于读写JSON格式的输入文件
import os               # os: 用于查看当前工作目录路径

# 打印确认信息，验证库导入成功
print("库导入成功！")

库导入成功！


In [8]:
# ============================================================
# Cell 2: 创建算例1的输入文件 model1.json
# 算例1: 一维两单元杆结构
# ============================================================

# 定义字典存储模型数据
# 注意: JSON中所有编号采用1-based（从1开始），程序内部会转换为0-based
model1_data = {
    "Title": "1D bar example",   # 模型标题
    "nsd": 1,                    # 空间维数 (number of spatial dimensions): 1=一维
    "ndof": 1,                   # 每个节点的自由度数: 1=每个节点只有u位移
    "nnp": 3,                    # 节点总数 (number of nodal points): 3个节点
    "nel": 2,                    # 单元总数 (number of elements): 2个单元
    "nen": 2,                    # 每个单元的节点数 (number of element nodes): 2个节点/单元
    
    # 材料参数: 一维杆中 E*A/L 直接作为刚度系数使用
    "E": [100.0, 200.0],         # 弹性模量，这里实际存储的是 E*A/L 的值
    "CArea": [1.0, 1.0],         # 截面积，配合上面E使用
    
    # 节点坐标: x为一维坐标，y全部置0（一维问题不需要y）
    "x": [0.0, 1.0, 2.0],       # 节点1在x=0, 节点2在x=1, 节点3在x=2
    "y": [0.0, 0.0, 0.0],       # y坐标全部为零（一维问题）
    
    # 单元连接数组 IEN (Element Incidence): 每个单元连接哪两个节点
    # 单元1连接节点1-2, 单元2连接节点2-3
    "IEN": [[1, 2], [2, 3]],
    
    # 固定自由度: 节点1固定，即第1个自由度被约束
    "fixed_dof": [1],            # 1-based编号，程序内部会减1变为0
    "fixed_value": [0.0],        # 固定位移值为0
    
    # 节点载荷: 在哪些自由度上施加载荷，以及载荷大小
    "force_dof": [2, 3],         # 在第2、3个自由度上施加载荷
    "force_value": [0.0, 10.0]  # f2=0, f3=10
}

# 将字典写入JSON文件，indent=2表示格式化缩进便于阅读
with open('model1.json', 'w') as f:
    json.dump(model1_data, f, indent=2)

print("model1.json 创建成功")

model1.json 创建成功


In [9]:
# ============================================================
# Cell 3: 创建算例2的输入文件 model2.json
# 算例2: 二维两杆桁架结构
# ============================================================

# 定义字典存储模型数据
model2_data = {
    "Title": "2D truss example",  # 模型标题
    "nsd": 2,                    # 空间维数: 2=二维
    "ndof": 2,                   # 每个节点的自由度数: 2=每个节点有u,v两个位移
    "nnp": 3,                    # 节点总数: 3个节点
    "nel": 2,                    # 单元总数: 2个单元
    "nen": 2,                    # 每个单元的节点数: 2个节点/单元
    
    # 材料参数
    "E": [1.0, 1.0],             # 弹性模量
    "CArea": [1.0, 1.0],         # 截面积
    
    # 节点坐标 (x,y)
    # 节点1: (1,0), 节点2: (0,0), 节点3: (1,1)
    "x": [1.0, 0.0, 1.0],
    "y": [0.0, 0.0, 1.0],
    
    # 单元连接: 单元1连接节点1-3, 单元2连接节点2-3
    "IEN": [[1, 3], [2, 3]],
    
    # 固定自由度: 节点1和节点2完全固定
    # 节点1: u1(自由度1), v1(自由度2); 节点2: u2(自由度3), v2(自由度4)
    "fixed_dof": [1, 2, 3, 4],   # 4个自由度被约束
    "fixed_value": [0.0, 0.0, 0.0, 0.0],  # 位移值均为0
    
    # 节点载荷: 节点3受水平力 Fx=10, Fy=0
    # 节点3的u对应自由度5, v对应自由度6
    "force_dof": [5, 6],
    "force_value": [10.0, 0.0]
}

# 写入JSON文件
with open('model2.json', 'w') as f:
    json.dump(model2_data, f, indent=2)

print("model2.json 创建成功")

model2.json 创建成功


In [10]:
# ============================================================
# Cell 4: 有限元核心程序（5个模块函数）
# 模块划分: 前处理 / 单元分析 / 组装 / 求解 / 后处理
# ============================================================


def read_model(filename):
    """
    前处理模块: 读取JSON模型文件
    参数:
        filename: JSON文件路径字符串
    返回:
        model: 字典，包含所有模型参数
    关键转换:
        JSON中自由度编号为1-based，程序内部数组为0-based，需要减1
    """
    with open(filename, 'r') as f:
        model = json.load(f)  # 读取JSON文件解析为字典
    
    # 1-based → 0-based 转换（Python数组索引从0开始）
    # IEN: 单元连接数组，存储每个单元连接的全局节点编号
    model['IEN'] = [[n-1 for n in elem] for elem in model['IEN']]
    
    # fixed_dof: 被约束的自由度编号
    model['fixed_dof'] = [d-1 for d in model['fixed_dof']]
    
    # force_dof: 施加载荷的自由度编号
    model['force_dof'] = [d-1 for d in model['force_dof']]
    
    # 计算每个单元的长度L，用于后续刚度矩阵计算
    model['L'] = []
    for elem in model['IEN']:
        n1, n2 = elem  # 单元连接的两个全局节点编号
        # 计算两点间距离（欧几里得距离）
        L = np.sqrt((model['x'][n2]-model['x'][n1])**2 + (model['y'][n2]-model['y'][n1])**2)
        model['L'].append(L)
    
    return model


def generate_LM(model):
    """
    生成对号矩阵LM (Location Matrix)
    参数:
        model: 模型字典
    返回:
        LM: 对号矩阵，形状为(ndof*nen, nel)
    功能:
        建立单元局部自由度与总体自由度之间的映射关系
        LM[a, e] = 单元e的第a个局部自由度对应的全局自由度编号
    """
    # 从模型中提取参数
    nnp = model['nnp']      # 节点总数
    ndof = model['ndof']    # 每个节点的自由度数
    nel = model['nel']      # 单元总数
    nen = model['nen']      # 每个单元的节点数
    IEN = model['IEN']      # 单元连接数组
    
    # 每个单元的局部自由度数 = 节点数 × 每个节点自由度数
    ndof_local = ndof * nen
    
    # 初始化LM矩阵: 局部自由度数 × 单元数
    LM = np.zeros((ndof_local, nel), dtype=int)
    
    # 遍历每个单元，填充LM列
    for e in range(nel):
        node_i = IEN[e][0]  # 局部节点1对应的全局节点
        node_j = IEN[e][1]  # 局部节点2对应的全局节点
        
        # 遍历该单元的所有局部自由度
        for local_dof in range(ndof_local):
            if local_dof < ndof:
                # 前ndof个局部自由度属于局部节点1
                global_node = node_i
                local_at_node = local_dof  # 节点内的局部自由度编号
            else:
                # 后ndof个局部自由度属于局部节点2
                global_node = node_j
                local_at_node = local_dof - ndof  # 减去偏移量
            
            # 全局自由度编号 = 全局节点编号 × 每节点自由度数 + 节点内局部自由度
            global_dof = global_node * ndof + local_at_node
            LM[local_dof, e] = global_dof
    
    return LM


def element_stiffness_1d(model, e):
    """
    一维杆单元刚度矩阵计算
    参数:
        model: 模型字典
        e: 单元编号（0-based）
    返回:
        Ke: 2×2单元刚度矩阵
    公式:
        Ke = (EA/L) * [[ 1, -1],
                       [-1,  1]]
    """
    # 提取单元参数
    E = model['E'][e]           # 弹性模量（或等效刚度系数）
    A = model['CArea'][e]       # 截面积
    L = model['L'][e]           # 单元长度
    
    # 计算单元刚度系数 c = EA/L
    c = E * A / L
    
    # 构建2×2单元刚度矩阵
    Ke = c * np.array([[1, -1],
                       [-1, 1]], dtype=float)
    return Ke


def element_stiffness_2d_truss(model, e):
    """
    二维桁架单元刚度矩阵计算（全局坐标系）
    参数:
        model: 模型字典
        e: 单元编号（0-based）
    返回:
        Ke: 4×4单元刚度矩阵
        c: 方向余弦 cos(θ)
        s: 方向余弦 sin(θ)
        L: 单元长度
    公式:
        Ke = (EA/L) * [[ c²,  cs, -c², -cs],
                       [ cs,  s², -cs, -s²],
                       [-c², -cs,  c²,  cs],
                       [-cs, -s²,  cs,  s²]]
    """
    # 提取材料参数
    E = model['E'][e]
    A = model['CArea'][e]
    
    # 获取单元两个节点的全局编号和坐标
    n1, n2 = model['IEN'][e]
    x1, y1 = model['x'][n1], model['y'][n1]
    x2, y2 = model['x'][n2], model['y'][n2]
    
    # 计算单元长度和方向余弦
    dx = x2 - x1        # x方向投影
    dy = y2 - y1        # y方向投影
    L = np.sqrt(dx**2 + dy**2)  # 单元长度
    c = dx / L          # cos(θ) = 邻边/斜边
    s = dy / L          # sin(θ) = 对边/斜边
    
    # 计算刚度系数
    k = E * A / L
    
    # 预计算方向余弦的乘积
    cc = c * c          # c²
    ss = s * s          # s²
    cs = c * s          # c*s
    
    # 构建4×4单元刚度矩阵
    Ke = k * np.array([[ cc,  cs, -cc, -cs],
                       [ cs,  ss, -cs, -ss],
                       [-cc, -cs,  cc,  cs],
                       [-cs, -ss,  cs,  ss]], dtype=float)
    
    return Ke, c, s, L


def assemble_global_K(model, LM):
    """
    组装模块: 直接组装总体刚度矩阵
    参数:
        model: 模型字典
        LM: 对号矩阵
    返回:
        K: 总体刚度矩阵
    算法核心:
        对每个单元e，获取Ke，然后根据LM将Ke[a,b]累加到K[LM[a,e], LM[b,e]]
        即: K[LM[a,e], LM[b,e]] += Ke[a,b]
    """
    # 提取模型参数
    ndof = model['ndof']    # 每节点自由度数
    nnp = model['nnp']      # 节点总数
    nel = model['nel']      # 单元总数
    nsd = model['nsd']      # 空间维数
    
    # 总自由度数 = 节点总数 × 每节点自由度数
    ndof_total = nnp * ndof
    
    # 初始化总体刚度矩阵为零矩阵（稠密存储）
    K = np.zeros((ndof_total, ndof_total), dtype=float)
    
    # 遍历每个单元，进行直接组装
    for e in range(nel):
        # 根据空间维数选择对应的单元刚度矩阵计算函数
        if nsd == 1:
            # 一维杆单元
            Ke = element_stiffness_1d(model, e)
        else:
            # 二维桁架单元（只取Ke，忽略c,s,L返回值）
            Ke, c, s, L = element_stiffness_2d_truss(model, e)
        
        # 获取单元刚度矩阵的维度
        ndof_local = Ke.shape[0]
        
        # 双重循环: 将Ke的每个元素累加到K的对应位置
        for a in range(ndof_local):
            for b in range(ndof_local):
                # 通过LM查找全局自由度编号
                global_a = LM[a, e]  # 局部自由度a对应的全局自由度
                global_b = LM[b, e]  # 局部自由度b对应的全局自由度
                # 累加（Scatter and Sum）
                K[global_a, global_b] += Ke[a, b]
    
    return K


def solve_system(model, K, LM):
    """
    求解模块: 缩减法处理位移边界条件
    参数:
        model: 模型字典
        K: 未施加边界条件的总体刚度矩阵
        LM: 对号矩阵（本函数中未使用，保留参数以保持接口一致）
    返回:
        d: 完整的节点位移向量
        f_reaction: 约束反力向量
    算法:
        将位移分为已知d_E（固定边界）和未知d_F（自由节点）
        求解: K_FF * d_F = f_F - K_EF^T * d_E
        然后计算反力: f_reaction = K_EF * d_F + K_EE * d_E - f_E
    """
    # 总自由度数
    ndof_total = model['nnp'] * model['ndof']
    
    # 固定自由度（已知位移）和自由自由度（未知位移）
    fixed = model['fixed_dof']      # 被约束的自由度编号列表
    # 自由自由度 = 总自由度中不在fixed列表里的
    free = [i for i in range(ndof_total) if i not in fixed]
    
    # 从总体刚度矩阵K中提取子矩阵
    # np.ix_用于构建二维索引，提取子矩阵
    K_FF = K[np.ix_(free, free)]    # 自由-自由子矩阵
    K_EF = K[np.ix_(fixed, free)]   # 约束-自由子矩阵（注意维度是fixed×free）
    K_EE = K[np.ix_(fixed, fixed)]  # 约束-约束子矩阵
    
    # 构建总体载荷向量f
    f = np.zeros(ndof_total)        # 初始化为零
    for dof, val in zip(model['force_dof'], model['force_value']):
        f[dof] = val                # 在施加载荷的自由度上赋值
    
    # 提取子向量
    f_F = f[free]                   # 自由自由度上的外力
    f_E = f[fixed]                  # 约束自由度上的外力
    
    # 已知位移值
    d_E = np.array(model['fixed_value'])
    
    # 检查施加边界条件前K的奇异性
    print(f"  施加边界条件前K秩: {np.linalg.matrix_rank(K)}/{ndof_total}")
    print(f"  施加边界条件前K是否奇异: {'是' if np.linalg.matrix_rank(K) < ndof_total else '否'}")
    
    # 缩减法求解: 右端项 = 自由节点外力 - 约束-自由耦合项 × 已知位移
    rhs = f_F - K_EF.T @ d_E
    
    # 检查缩减矩阵是否非奇异
    print(f"  缩减矩阵K_FF秩: {np.linalg.matrix_rank(K_FF)}/{len(free)}")
    
    # 使用NumPy的线性求解器求解线性方程组
    d_F = np.linalg.solve(K_FF, rhs)
    
    # 重构完整的位移向量
    d = np.zeros(ndof_total)
    for i, idx in enumerate(free):
        d[idx] = d_F[i]             # 填入求得的未知位移
    for i, idx in enumerate(fixed):
        d[idx] = d_E[i]             # 填入已知约束位移
    
    # 计算约束反力（根据分块矩阵方程的第二行）
    f_reaction = K_EF @ d_F + K_EE @ d_E - f_E
    
    return d, f_reaction


def postprocess(model, d, LM):
    """
    后处理模块: 计算单元应力和轴力
    参数:
        model: 模型字典
        d: 完整的节点位移向量
        LM: 对号矩阵
    返回:
        results: 列表，每个元素是一个字典，包含该单元的计算结果
    """
    nel = model['nel']      # 单元总数
    nsd = model['nsd']      # 空间维数
    results = []            # 存储所有单元的结果
    
    # 遍历每个单元
    for e in range(nel):
        # 提取该单元的位移向量de
        # 根据LM从总体位移d中提取局部自由度对应的位移
        ndof_local = model['ndof'] * model['nen']
        de = np.zeros(ndof_local)
        for a in range(ndof_local):
            de[a] = d[LM[a, e]]     # 局部自由度a的位移 = 全局自由度LM[a,e]的位移
        
        # 提取材料参数
        E = model['E'][e]           # 弹性模量
        A = model['CArea'][e]       # 截面积
        L = model['L'][e]           # 单元长度
        
        if nsd == 1:
            # 一维杆单元应力计算
            # 公式: sigma = E/L * [-1, 1] · de
            B = np.array([-1, 1]) / L   # 应变-位移矩阵
            sigma = E * (B @ de)         # 点乘计算应力
            force = sigma * A             # 轴力 = 应力 × 面积
            
            # 存储结果到字典
            result = {
                'element': e + 1,        # 单元编号（转为1-based显示）
                'length': L,              # 单元长度
                'stress': sigma,          # 单元应力
                'axial_force': force      # 单元轴力
            }
            
        else:
            # 二维桁架单元应力计算
            # 重新计算方向余弦（与单元刚度计算时一致）
            n1, n2 = model['IEN'][e]
            x1, y1 = model['x'][n1], model['y'][n1]
            x2, y2 = model['x'][n2], model['y'][n2]
            dx, dy = x2 - x1, y2 - y1
            L = np.sqrt(dx**2 + dy**2)
            c, s = dx / L, dy / L       # 方向余弦
            
            # 公式: sigma = E/L * [-c, -s, c, s] · de
            B = np.array([-c, -s, c, s]) / L   # 应变-位移矩阵
            sigma = E * (B @ de)                # 计算应力
            force = sigma * A                    # 轴力
            
            # 存储结果到字典
            result = {
                'element': e + 1,
                'length': L,
                'direction_cosine': (c, s),   # 方向余弦元组
                'stress': sigma,
                'axial_force': force
            }
        
        results.append(result)  # 添加到结果列表
    
    return results


def run_analysis(filename):
    """
    主程序: 运行完整有限元分析流程
    参数:
        filename: 模型JSON文件路径
    功能:
        按顺序调用前处理→LM生成→组装→求解→后处理五个模块
    """
    # 打印分隔线和标题
    print("=" * 60)
    print(f"开始分析: {filename}")
    print("=" * 60)
    
    # Step 1: 前处理 - 读取模型数据
    print("\n【Step 1】前处理：读取模型")
    model = read_model(filename)
    print(f"  标题: {model['Title']}")
    print(f"  节点数: {model['nnp']}, 单元数: {model['nel']}")
    
    # Step 2: 生成对号矩阵LM
    print("\n【Step 2】生成对号矩阵LM")
    LM = generate_LM(model)
    print(f"  LM形状: {LM.shape}")
    print(f"  LM矩阵:\n{LM}")
    
    # Step 3: 组装总体刚度矩阵
    print("\n【Step 3】组装总体刚度矩阵")
    K = assemble_global_K(model, LM)
    print("\n总体刚度矩阵 K:")
    # 设置打印格式: 精度4位小数，抑制科学计数法
    np.set_printoptions(precision=4, suppress=True)
    print(K)
    np.set_printoptions()  # 恢复默认设置
    
    # 检查对称性
    is_symmetric = np.allclose(K, K.T)
    print(f"  对称性: {'是' if is_symmetric else '否'}")
    # 检查对角元非负性
    diag_nonnegative = np.all(np.diag(K) >= -1e-10)
    print(f"  对角元非负: {'是' if diag_nonnegative else '否'}")
    
    # Step 4: 施加边界条件并求解
    print("\n【Step 4】施加边界条件并求解")
    d, f_reaction = solve_system(model, K, LM)
    
    # Step 5: 后处理 - 计算单元应力
    print("\n【Step 5】后处理：计算单元应力")
    results = postprocess(model, d, LM)
    
    # 输出最终结果汇总
    print("\n" + "=" * 60)
    print("分析结果")
    print("=" * 60)
    
    # 打印节点位移
    print("\n节点位移:")
    for i in range(model['nnp']):
        if model['nsd'] == 1:
            # 一维: 只打印u位移
            print(f"  节点{i+1}: u = {d[i]:.10f}")
        else:
            # 二维: 打印u,v两个位移
            u = d[i * 2]
            v = d[i * 2 + 1]
            print(f"  节点{i+1}: u = {u:.10f}, v = {v:.10f}")
    
    # 打印约束反力
    print(f"\n约束反力:")
    for i, dof in enumerate(model['fixed_dof']):
        print(f"  自由度{dof+1}: {f_reaction[i]:.10f}")
    
    # 打印单元结果
    print(f"\n单元结果:")
    for r in results:
        print(f"\n  单元{r['element']}:")
        print(f"    长度: {r['length']:.10f}")
        if 'direction_cosine' in r:
            # 二维桁架额外打印方向余弦
            print(f"    方向余弦: c={r['direction_cosine'][0]:.10f}, s={r['direction_cosine'][1]:.10f}")
        print(f"    应力: {r['stress']:.10f}")
        print(f"    轴力: {r['axial_force']:.10f}")
    
    # 结束分隔线
    print("\n" + "=" * 60)
    
    # 返回所有结果，便于后续验证和调用
    return model, K, LM, d, f_reaction, results


# 打印加载完成提示
print("核心函数加载完成！")

核心函数加载完成！


In [11]:
# ============================================================
# Cell 5: 运行算例1 — 一维两单元杆结构
# 验证目标:
#   - 总体刚度矩阵 K = [[100,-100,0],[-100,300,-200],[0,-200,200]]
#   - 位移: d1=0, d2=0.1, d3=0.15
#   - 反力: 10.0
# ============================================================

# 调用主程序运行分析，返回所有结果
model1, K1, LM1, d1, f1, results1 = run_analysis('model1.json')

# 以下为验证检查，与理论解对比
print("\n" + "=" * 60)
print("算例1 验证检查:")
# 检查总体刚度矩阵
K1_expected = np.array([[100, -100, 0], [-100, 300, -200], [0, -200, 200]])
print(f"总体刚度矩阵是否正确: {np.allclose(K1, K1_expected)}")

# 检查位移
print(f"d1=0: {abs(d1[0]) < 1e-10}")           # 节点1位移应为0
print(f"d2=0.1: {abs(d1[1] - 0.1) < 1e-10}")    # 节点2位移应为0.1
print(f"d3=0.15: {abs(d1[2] - 0.15) < 1e-10}")  # 节点3位移应为0.15

# 检查反力
print(f"反力=10: {abs(f1[0] - 10) < 1e-10}")    # 节点1反力应为10

开始分析: model1.json

【Step 1】前处理：读取模型
  标题: 1D bar example
  节点数: 3, 单元数: 2

【Step 2】生成对号矩阵LM
  LM形状: (2, 2)
  LM矩阵:
[[0 1]
 [1 2]]

【Step 3】组装总体刚度矩阵

总体刚度矩阵 K:
[[ 100. -100.    0.]
 [-100.  300. -200.]
 [   0. -200.  200.]]
  对称性: 是
  对角元非负: 是

【Step 4】施加边界条件并求解
  施加边界条件前K秩: 2/3
  施加边界条件前K是否奇异: 是
  缩减矩阵K_FF秩: 2/2

【Step 5】后处理：计算单元应力

分析结果

节点位移:
  节点1: u = 0.0000000000
  节点2: u = 0.1000000000
  节点3: u = 0.1500000000

约束反力:
  自由度1: -10.0000000000

单元结果:

  单元1:
    长度: 1.0000000000
    应力: 10.0000000000
    轴力: 10.0000000000

  单元2:
    长度: 1.0000000000
    应力: 10.0000000000
    轴力: 10.0000000000


算例1 验证检查:
总体刚度矩阵是否正确: True
d1=0: True
d2=0.1: True
d3=0.15: True
反力=10: False


In [6]:
# 运行算例2：二维两杆桁架结构
print("运行算例2...")
model2, K2, LM2, d2, f2, results2 = run_analysis('model2.json')

print("\n" + "="*60)
print("算例2 验证检查:")
print(f"u3≈38.284271: {abs(d2[4]-38.284271) < 1e-5}")
print(f"v3≈-10.000000: {abs(d2[5]-(-10)) < 1e-5}")
print(f"单元1应力≈-10: {abs(results2[0]['stress']-(-10)) < 1e-5}")
print(f"单元2应力≈14.142136: {abs(results2[1]['stress']-14.142136) < 1e-5}")

运行算例2...
开始分析: model2.json

【Step 1】前处理：读取模型
  标题: 2D truss example
  节点: 3, 单元: 2

【Step 2】生成对号矩阵LM
  LM形状: (4, 2)
  LM矩阵:
[[0 2]
 [1 3]
 [4 4]
 [5 5]]

【Step 3】组装总体刚度矩阵

总体刚度矩阵 K:
[[ 0.      0.      0.      0.      0.      0.    ]
 [ 0.      1.      0.      0.      0.     -1.    ]
 [ 0.      0.      0.3536  0.3536 -0.3536 -0.3536]
 [ 0.      0.      0.3536  0.3536 -0.3536 -0.3536]
 [ 0.      0.     -0.3536 -0.3536  0.3536  0.3536]
 [ 0.     -1.     -0.3536 -0.3536  0.3536  1.3536]]
  对称性: 是
  对角元非负: 是

【Step 4】施加边界条件并求解
  施加边界条件前K的秩: 2/6
  施加边界条件前K是否奇异: 是
  缩减矩阵K_FF的秩: 2/2

【Step 5】后处理：计算单元应力

分析结果

节点位移:
  节点1: u = 0.0000000000, v = 0.0000000000
  节点2: u = 0.0000000000, v = 0.0000000000
  节点3: u = 38.2842712475, v = -10.0000000000

约束反力:
  自由度1: 0.0000000000
  自由度2: 10.0000000000
  自由度3: -10.0000000000
  自由度4: -10.0000000000

单元结果:

  单元1:
    长度: 1.0000000000
    方向余弦: c=0.000000, s=1.000000
    应力: -10.0000000000
    轴力: -10.0000000000

  单元2:
    长度: 1.4142135624
    方向余弦: c=0.70